In [0]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    sum as _sum,
    lit
)
from delta.tables import DeltaTable

# =====================================================
# WIDGET PARAMETER (Workflow Input)
# =====================================================

dbutils.widgets.text("arrival_date", "2024-07-26")

date_str = dbutils.widgets.get("arrival_date") #arrival_date from jobflow as key is 2024-07-26 

print(f"Processing Date : {date_str}")

# =====================================================
# FILE PATHS
# =====================================================

booking_data = f"/Volumes/incremental_load/default/orders_data/booking_data/bookings_{date_str}.csv"

customer_data = f"/Volumes/incremental_load/default/orders_data/customer_data/customers_{date_str}.csv"

print(booking_data)
print(customer_data)

# =====================================================
# READ BOOKING FILE
# =====================================================

booking_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", "\"")
    .option("multiLine", "true")
    .load(booking_data)
)

# =====================================================
# READ CUSTOMER FILE
# =====================================================

customer_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", "\"")
    .option("multiLine", "true")
    .load(customer_data)
)

print("Files loaded successfully")

# =====================================================
# DATA QUALITY CHECKS - BOOKINGS
# =====================================================

if booking_df.count() == 0:
    raise Exception("Booking file is empty")

if booking_df.filter(col("booking_id").isNull()).count() > 0:
    raise Exception("booking_id contains NULL values")

if booking_df.filter(col("customer_id").isNull()).count() > 0:
    raise Exception("customer_id contains NULL values")

if booking_df.filter(col("amount").isNull()).count() > 0:
    raise Exception("amount contains NULL values")

if booking_df.filter(col("amount") < 0).count() > 0:
    raise Exception("amount contains negative values")

if booking_df.filter(col("quantity") < 0).count() > 0:
    raise Exception("quantity contains negative values")

if booking_df.filter(col("discount") < 0).count() > 0:
    raise Exception("discount contains negative values")

duplicate_booking_ids = (
    booking_df
    .groupBy("booking_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

if duplicate_booking_ids > 0:
    raise Exception("Duplicate booking_id found")

print("Booking DQ Checks Passed")

# =====================================================
# DATA QUALITY CHECKS - CUSTOMERS
# =====================================================

if customer_df.count() == 0:
    raise Exception("Customer file is empty")

if customer_df.filter(col("customer_id").isNull()).count() > 0:
    raise Exception("customer_id contains NULL values")

if customer_df.filter(col("customer_name").isNull()).count() > 0:
    raise Exception("customer_name contains NULL values")

if customer_df.filter(col("customer_address").isNull()).count() > 0:
    raise Exception("customer_address contains NULL values")

if customer_df.filter(col("email").isNull()).count() > 0:
    raise Exception("email contains NULL values")

duplicate_customer_ids = (
    customer_df
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

if duplicate_customer_ids > 0:
    raise Exception("Duplicate customer_id found")

print("Customer DQ Checks Passed")

# =====================================================
# ADD INGESTION TIMESTAMP
# =====================================================

booking_df_incremental = booking_df.withColumn(
    "ingestion_time",
    current_timestamp()
)

# =====================================================
# JOIN CUSTOMER + BOOKING
# =====================================================

df_joined = booking_df_incremental.join(
    customer_df,
    "customer_id"
)

# =====================================================
# BUSINESS TRANSFORMATION
# =====================================================

df_transformed = (
    df_joined
    .withColumn(
        "total_cost",
        col("amount") - col("discount")
    )
    .filter(col("quantity") > 0)
)

# =====================================================
# AGGREGATION
# =====================================================

df_transformed_agg = (
    df_transformed
    .groupBy(
        "booking_type",
        "customer_id"
    )
    .agg(
        _sum("total_cost").alias("total_amount_sum"),
        _sum("quantity").alias("total_quantity_sum")
    )
)

display(df_transformed_agg)

# =====================================================
# FACT TABLE
# =====================================================

fact_table_name = "incremental_load.default.booking_fact"

if not spark.catalog.tableExists(fact_table_name):

    (
        df_transformed_agg.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(fact_table_name)
    )

    print("booking_fact created")

else:

    fact_delta = DeltaTable.forName(
        spark,
        fact_table_name
    )

    (
        fact_delta.alias("target")
        .merge(
            df_transformed_agg.alias("source"),
            """
            target.booking_type = source.booking_type
            AND
            target.customer_id = source.customer_id
            """
        )
        .whenMatchedUpdate(
            set={
                "total_amount_sum":
                    "target.total_amount_sum + source.total_amount_sum",

                "total_quantity_sum":
                    "target.total_quantity_sum + source.total_quantity_sum"
            }
        )
        .whenNotMatchedInsert(
            values={
                "booking_type":
                    "source.booking_type",

                "customer_id":
                    "source.customer_id",

                "total_amount_sum":
                    "source.total_amount_sum",

                "total_quantity_sum":
                    "source.total_quantity_sum"
            }
        )
        .execute()
    )

    print("booking_fact merged successfully")

display(
    spark.table(fact_table_name)
)

# =====================================================
# CUSTOMER DIMENSION SCD2
# =====================================================

scd_table_name = "incremental_load.default.customer_dim"

customer_updates = (
    customer_df
    .withColumn(
        "valid_from",
        current_timestamp()
    )
    .withColumn(
        "valid_to",
        lit("9999-12-31")
    )
)

if not spark.catalog.tableExists(scd_table_name):

    (
        customer_updates.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(scd_table_name)
    )

    print("customer_dim created")

else:

    scd_delta = DeltaTable.forName(
        spark,
        scd_table_name
    )

    (
        scd_delta.alias("target")
        .merge(
            customer_updates.alias("source"),
            """
            target.customer_id = source.customer_id
            AND target.valid_to = TIMESTAMP('9999-12-31')
            """
        )
        .whenMatchedUpdate(
            condition="""
                target.customer_name <> source.customer_name
                OR target.customer_address <> source.customer_address
                OR target.email <> source.email
            """,
            set={
                "valid_to": "CAST(current_timestamp() AS STRING)"
            }
        )
        .execute()
    )

    active_records = (
        customer_updates.alias("source")
        .join(
            spark.table(scd_table_name)
            .alias("target"),
            "customer_id",
            "left"
        )
    )

    (
        customer_updates.write
        .format("delta")
        .mode("append")
        .saveAsTable(scd_table_name)
    )

    print("customer_dim SCD2 merge completed")

display(
    spark.table(scd_table_name)
)

print("PIPELINE COMPLETED SUCCESSFULLY")

In [0]:
spark.table("incremental_load.default.customer_dim").printSchema()